#  Notebook 1 — EDA & Limpieza
## HMDA New York 2024: Sesgo, Equidad y Explicabilidad en Hipotecas

**Autores:** Izan Cuesta Corbí · Dennis García Solera · Marcos Segurado Llopis · Jesús Cano Moya  
**Dataset:** Home Mortgage Disclosure Act (HMDA) — New York 2024 (CFPB)

---
> **Objetivo de este notebook:** Exploración profunda del dataset, detección de sesgos en atributos sensibles (raza, sexo, etnia y edad), limpieza y preparación de los datos para los modelos posteriores.

> `RANDOM_STATE = 42` fijado globalmente para reproducibilidad.

---
## IMPORTS

In [1]:
import sys
import os
import warnings
import scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

np.random.seed(42)
RANDOM_STATE = 42

---
## Carga del dataset

El dataset HMDA (Home Mortgage Disclosure Act) es publicado anualmente por la CFPB (Consumer Financial Protection Bureau) de EEUU. Contiene todas las solicitudes de hipoteca registradas en Nueva York durante 2024, incluyendo datos demográficos del solicitante, características del préstamo y la decisión final de la entidad financiera.

In [2]:
df_original = pd.read_csv('../data/state_NY.csv', low_memory=False)

print(f"Dataset cargado: {df_original.shape[0]:,} instancias, {df_original.shape[1]} columnas")
print(f"\nPrimeras 3 filas:")
df_original.head(3)

Dataset cargado: 383,577 instancias, 99 columnas

Primeras 3 filas:


,activity_year,lei,derived_msa-md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason-2,denial_reason-3,denial_reason-4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2024,549300WEZMN6QE5IIH42,35614,NY,36061.0,3.606102e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,9383,38.70,101900,253,1252,253,0
1,2024,549300WEZMN6QE5IIH42,10580,NY,36083.0,3.608304e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,1731,31.95,117800,64,234,773,0
2,2024,549300WEZMN6QE5IIH42,35614,NY,36061.0,3.606101e+10,C,Conventional:First Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4481,27.92,101900,257,109,40,63


### Observaciones

- El dataset contiene **383.577 solicitudes de hipoteca** en Nueva York durante
  2024, con 99 columnas que cubren características del préstamo, del solicitante
  y datos demográficos.

- Los códigos string de raza, sexo y etnia son **atributos sensibles explícitos**,
  a diferencia de otros datasets donde están codificados indirectamente.
  
- El target `action_taken` tiene 8 valores posibles. Para nuestro problema de
  clasificación binaria nos quedamos únicamente con:
  - `1` = Préstamo concedido 
  - `3` = Solicitud denegada 


In [3]:
df_base = df_original[df_original['action_taken'].isin([1,3])].copy()

# Recodificar target: 1 = concedido, 0 = denegado
df_base['action_taken'] = df_base['action_taken'].map({1: 1, 3: 0})

df_base.to_csv('../data/hmda_base.csv')

---
## Exploración del dataset

In [4]:
print("=" * 55)
print("  SHAPE")
print("=" * 55)
print(f"  Filas:    {df_base.shape[0]:,}")
print(f"  Columnas: {df_base.shape[1]}")

print("\n" + "=" * 55)
print("  TIPOS DE DATOS")
print("=" * 55)
print(df_base.dtypes.to_string())

print("\n" + "=" * 55)
print("  VALORES FALTANTES (columnas con >0 nulos)")
print("=" * 55)

faltantes = df_base.isnull().sum()
faltantes_df = pd.DataFrame({'N nulos': faltantes, '% nulos': round(faltantes / len(df_base) * 100,2)})
faltantes_df = faltantes_df[faltantes_df['N nulos'] > 0].sort_values('% nulos', ascending=False)

print(faltantes_df.to_string())
print(f"\n  Total columnas con nulos: {len(faltantes_df)}")
print(f"  Total columnas sin nulos: {df_base.shape[1] - len(faltantes_df)}")

  SHAPE
  Filas:    283,332
  Columnas: 99

  TIPOS DE DATOS
activity_year                                 int64
lei                                          object
derived_msa-md                                int64
state_code                                   object
county_code                                 float64
census_tract                                float64
conforming_loan_limit                        object
derived_loan_product_type                    object
derived_dwelling_category                    object
derived_ethnicity                            object
derived_race                                 object
derived_sex                                  object
action_taken                                  int64
purchaser_type                                int64
preapproval                                   int64
loan_type                                     int64
loan_purpose                                  int64
lien_status                                   int64
rev